In [3]:
import pandas as pd
df = pd.read_csv("/Users/colinclapper/Desktop/hiv-resistance-predictor/hiv-resistance-predictor/data/raw/PI_DataSet.txt", sep = "\t")

In [5]:
# view the data
print(df.shape)
print(df.columns.tolist())
print(df.head())

(2171, 109)
['SeqID', 'FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P46', 'P47', 'P48', 'P49', 'P50', 'P51', 'P52', 'P53', 'P54', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62', 'P63', 'P64', 'P65', 'P66', 'P67', 'P68', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P75', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P82', 'P83', 'P84', 'P85', 'P86', 'P87', 'P88', 'P89', 'P90', 'P91', 'P92', 'P93', 'P94', 'P95', 'P96', 'P97', 'P98', 'P99', 'CompMutList']
   SeqID  FPV  ATV  IDV  LPV   NFV  SQV  TPV  DRV P1  ... P91 P92 P93 P94 P95  \
0  12862  0.8  NaN  1.2  NaN  24.7  0.9  NaN  NaN  -  ...   -   -   -   -   -   
1  13259  0.1  NaN  2.9  NaN  12.2  1.4  NaN  NaN  -  ...   -

In [ ]:
# Find which drugs have missing rows/columns.
df[['FPV', 'ATV', 'IDV', 'LPV', 'NFV', 'SQV', 'TPV', 'DRV']].isna().sum()

FPV     119
ATV     666
IDV      73
LPV     364
NFV      38
SQV      87
TPV     945
DRV    1178
dtype: int64

In [13]:
#drop rows with missing values in NFV column
df['NFV'].isna()
nfv_df = df[df['NFV'].notna()].copy()
#print shape of new dataframe
print(nfv_df.shape)
#describe the new dataframe
print(nfv_df.describe()) 

(2133, 109)
               SeqID          FPV          ATV          IDV          LPV  \
count    2133.000000  2049.000000  1500.000000  2094.000000  1800.000000   
mean   185706.541960    14.242899    24.131933    16.953486    28.197444   
std    204608.605534    26.303997    34.853692    27.452641    37.691065   
min      2996.000000     0.100000     0.300000     0.100000     0.100000   
25%     56142.000000     0.700000     1.000000     0.900000     0.800000   
50%    109400.000000     1.800000     3.850000     2.950000     4.000000   
75%    205638.000000    13.000000    33.000000    20.000000    52.000000   
max    615540.000000   100.000000   100.000000   100.000000   100.000000   

               NFV          SQV         TPV         DRV  
count  2133.000000  2083.000000  1194.00000  963.000000  
mean     24.286451    20.453145     6.48124   11.797196  
std      32.647711    34.127542    18.77293   26.044970  
min       0.100000     0.100000     0.10000    0.200000  
25%       1.2

In [15]:
#Create new binary column for NFV resistance, where 1 if NFV >= 3.6 else 0
nfv_df['resistance'] = nfv_df['NFV'].apply(lambda x: 1 if x>= 3.6 else 0)
#Check class balance of new binary column
nfv_df['resistance'].value_counts()

resistance
1    1182
0     951
Name: count, dtype: int64

In [17]:
print(nfv_df['P30'].value_counts())
print(nfv_df['P90'].value_counts())

P30
-    2021
N     112
Name: count, dtype: int64
P90
-      1386
M       701
LM       35
.         5
MV        1
IM        1
V         1
LW        1
LIM       1
LS        1
Name: count, dtype: int64


In [ ]:
#across all 99 position columns, how many values exist in each one  and only prints the columns where it's actually a nonzero problem.
p_columns = [f'P{i}' for i in range(1, 100)]
missing_counts = (nfv_df[p_columns] == '.').sum()
print(missing_counts[missing_counts > 0])

P1     226
P2     226
P3     225
P4      38
P5      31
P6      25
P7      15
P8      13
P9       9
P10      8
P11      3
P12      2
P13      2
P14      1
P15      1
P16      1
P17      1
P18      1
P19      1
P20      1
P89      5
P90      5
P91      5
P92      5
P93      6
P94      8
P95      8
P96      8
P97      8
P98      9
P99     20
dtype: int64


In [ ]:
#decide wether dropping rows with missing values(no data on mutations) is a good idea or not.
total_rows_with_any_missing = (nfv_df[p_columns] == '.').any(axis=1).sum()
print(total_rows_with_any_missing)
print(nfv_df.shape[0])
print(2133-230)

230
2133


In [32]:
#clean the data and drop rows with missing values in any of the position columns
clean_df = nfv_df[(nfv_df[p_columns] != '.').all(axis=1)].copy()
print(clean_df.shape)
#drop column created earlier called NFV_resitance, accidentally created a new column called resistance, so dropping the old one.
clean_df.drop(columns=['NFV_resistance'], inplace=True)
clean_df.head()

(1903, 111)


,SeqID,FPV,ATV,IDV,LPV,NFV,SQV,TPV,DRV,P1,...,P92,P93,P94,P95,P96,P97,P98,P99,CompMutList,resistance
0,12862,0.8,NaN,1.2,NaN,24.7,0.9,NaN,NaN,-,...,-,-,-,-,-,-,-,-,"D30N, M46I, R57G, L63P, N88D",1
1,13259,0.1,NaN,2.9,NaN,12.2,1.4,NaN,NaN,-,...,-,-,-,-,-,-,-,-,"M46L, R57G, L63P, V77I, N88S",1
2,12863,3.0,NaN,2.8,NaN,2.2,1.0,NaN,NaN,-,...,-,-,-,-,-,-,-,-,"M46I, R57G, L63P, V82T, I84V",0
3,13257,0.1,NaN,2.3,NaN,8.1,1.1,NaN,NaN,-,...,-,-,-,-,-,-,-,-,"M46L, R57G, L63P, N88S",1
4,13258,0.1,NaN,3.0,NaN,12.8,1.2,NaN,NaN,-,...,-,-,-,-,-,-,-,-,"R57G, L63P, V77I, N88S",1


In [40]:
#Use comparison operator to check if the value is not equal to '-' and convert it to an integer. This will create a binary feature for each mutation.
feature_df = (clean_df[p_columns] != '-').astype(int)
feature_df.shape
feature_df.head()

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,...,P90,P91,P92,P93,P94,P95,P96,P97,P98,P99
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
#targeted check
print(feature_df.loc[0,'P30'])
print(feature_df["P30"].sum())
print(feature_df['P90'].sum())

1
103
672


In [49]:
#feature df created from clean df check
print(feature_df.index.equals(clean_df.index))
#build final matrix for model training, with features and target variable
X = feature_df
y = clean_df['resistance']
#final sanity check to make sure the number of rows in X and y are the same
print(X.shape)
print(y.shape)
print(y.value_counts())

True
(1903, 99)
(1903,)
resistance
1    1033
0     870
Name: count, dtype: int64


In [52]:
#save cleaned data to csv for future use
clean_df.to_csv("/Users/colinclapper/Desktop/hiv-resistance-predictor/hiv-resistance-predictor/data/processed/nfv_clean.csv", index=False)